# M1 Notebook 06 — Eigenvalues, Eigenvectors, and Spectral Intuition

**Notebook ID:** M1_N06  
**Status:** Runnable first edition  
**Random seed:** 42


## Learning objectives

1. Define eigenvalues and eigenvectors.
2. Verify eigenpairs numerically.
3. Understand diagonalization.
4. Estimate dominant eigenpairs with power iteration.
5. Analyze spectral radius and stability.
6. Connect spectral ideas to PCA, graphs, and Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.algebra import (
    diagonalize, eigendecomposition, matrix_power_via_eigendecomposition,
    power_iteration, spectral_radius, symmetric_eigendecomposition,
    verify_eigenpair,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()


## Definition

\[
A\mathbf v=\lambda\mathbf v.
\]

An eigenvector keeps its direction under the transformation; the eigenvalue records the scale factor.


In [ ]:
A=np.array([[3.,0.],[0.,1.]])
values,vectors=eigendecomposition(A)
for i,value in enumerate(values):
    assert verify_eigenpair(A,value,vectors[:,i])
values,vectors


## Geometric visualization

In [ ]:
grid=np.array([[-1,0],[0,1],[1,0],[0,-1],[1,1],[-1,1],[-1,-1],[1,-1]],float)
transformed=grid@A.T
fig,ax=plt.subplots(figsize=(6,6))
for original,new in zip(grid,transformed):
    ax.quiver(0,0,original[0],original[1],angles="xy",scale_units="xy",scale=1)
    ax.quiver(0,0,new[0],new[1],angles="xy",scale_units="xy",scale=1)
ax.axhline(0,linewidth=0.8); ax.axvline(0,linewidth=0.8)
ax.set_xlim(-4,4); ax.set_ylim(-4,4); ax.set_aspect("equal")
ax.set_title("Original and Transformed Vectors")
plt.show()


## Symmetric matrices

Real symmetric matrices have real eigenvalues and an orthonormal eigenbasis.


In [ ]:
S=np.array([[2.,1.],[1.,2.]])
s_values,s_vectors=symmetric_eigendecomposition(S)
assert np.allclose(s_values,[3.,1.])
assert np.allclose(s_vectors.T@s_vectors,np.eye(2))
s_values,s_vectors


## Diagonalization

\[
A=PDP^{-1}.
\]


In [ ]:
P,D,P_inv=diagonalize(S)
assert np.allclose(P@D@P_inv,S)
P,D


## Matrix powers

\[
A^k=PD^kP^{-1}.
\]


In [ ]:
k=10
spectral_power=matrix_power_via_eigendecomposition(S,k)
direct_power=np.linalg.matrix_power(S,k)
assert np.allclose(spectral_power,direct_power)
np.real_if_close(spectral_power)


## Power iteration

In [ ]:
dominant_value,dominant_vector,iterations=power_iteration(S)
assert np.isclose(dominant_value,3.,rtol=1e-8)
{"dominant_eigenvalue":dominant_value,"dominant_eigenvector":dominant_vector,"iterations":iterations}


## Spectral radius and stability

\[
\rho(A)=\max_i|\lambda_i|.
\]


In [ ]:
stable=np.array([[0.7,0.1],[0.,0.8]])
unstable=np.array([[1.1,0.],[0.,0.9]])
{"stable":spectral_radius(stable),"unstable":spectral_radius(unstable)}


In [ ]:
x0=np.array([1.,1.]); steps=20
stable_path=[x0]; unstable_path=[x0]
for _ in range(steps):
    stable_path.append(stable@stable_path[-1])
    unstable_path.append(unstable@unstable_path[-1])
stable_path=np.array(stable_path); unstable_path=np.array(unstable_path)
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(np.linalg.norm(stable_path,axis=1),label="Stable system")
ax.plot(np.linalg.norm(unstable_path,axis=1),label="Unstable system")
ax.set_xlabel("Time step"); ax.set_ylabel("State-vector norm")
ax.set_title("Spectral Radius and Dynamic Stability")
ax.legend()
plt.show()


## Statistics interpretation

Covariance eigenvectors define principal directions of variation; eigenvalues quantify variance along those directions.


In [ ]:
X=np.array([[2.,1.],[3.,2.],[4.,2.5],[5.,4.]])
Xc=X-X.mean(axis=0)
cov=np.cov(Xc,rowvar=False)
cov_values,cov_vectors=symmetric_eigendecomposition(cov)
pd.DataFrame({"eigenvalue":cov_values,"explained_variance_ratio":cov_values/cov_values.sum()})


## AI and network interpretation

Spectral methods support PCA, spectral clustering, graph neural networks, recurrent-system analysis, diffusion, PageRank, and low-rank models.


## Decision Intelligence case — Sector propagation modes

In [ ]:
sectors=["Agriculture","Health","Energy","Transport"]
M=np.array([
    [0.70,0.10,0.20,0.15],
    [0.10,0.80,0.25,0.10],
    [0.25,0.10,0.75,0.30],
    [0.20,0.15,0.35,0.70],
])
value,vector,iterations=power_iteration(M)
mode=np.abs(vector)/np.abs(vector).sum()
pd.Series(mode,index=sectors,name="dominant_mode_weight")


The dominant mode summarizes a long-run propagation direction in this illustrative linear model. It does not establish causal importance or policy priority.


## Engineering notes

- Non-symmetric matrices may have complex eigenvalues.
- Repeated eigenvalues may produce unstable eigenvectors.
- Power iteration identifies only a dominant mode.
- Large sparse systems require iterative eigensolvers.
- Eigenvectors are defined only up to scale and sign.


## Exercises

### Level A
Explain eigenvalues geometrically.

### Level B
Derive a characteristic polynomial for a 2-by-2 matrix.

### Level C
Implement power iteration.

### Capstone
Analyze the stability and dominant propagation mode of a validated sector-transition matrix.


## Key insight

Eigenvalues and eigenvectors expose invariant directions, dominant dynamics, variance structure, and stability hidden inside matrices.
